In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.svm import SVR

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import mlflow

c:\Users\rajpu\OneDrive\Desktop\ML_REAL_PROJECT\my-uber-demand prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import dagshub
dagshub.init(repo_owner='rajputriya887', repo_name='uber-demand-prediction', mlflow=True)

Repository uber-demand-prediction doesn't exist, creating it under current user.

Initialized MLflow to track repo "rajputriya887/uber-demand-prediction"

Repository rajputriya887/uber-demand-prediction initialized!

In [7]:
# set the dagshub tracking server

mlflow.set_tracking_uri("https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow")

In [ ]:
# load the training and test data

train_data_path = "../data/processed/train.csv"
test_data_path = "../data/processed/test.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

train_df

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,12.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


In [9]:
# missing value in training data

train_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [10]:
# missing values in the test data

test_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [11]:
# make X_train and y_train

X_train = train_df.drop(columns=["total_pickups"])

y_train = train_df["total_pickups"]

In [12]:
X_train.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185.0,4


In [13]:
# make X_test and y_test

X_test = test_df.drop(columns=["total_pickups"])

y_test = test_df["total_pickups"]

In [14]:
X_test.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-03-01 00:00:00,36.0,44.0,31.0,29.0,0,38.0,1
2016-03-01 00:15:00,41.0,36.0,44.0,31.0,0,39.0,1
2016-03-01 00:30:00,35.0,41.0,36.0,44.0,0,37.0,1
2016-03-01 00:45:00,47.0,35.0,41.0,36.0,0,41.0,1
2016-03-01 01:00:00,34.0,47.0,35.0,41.0,0,38.0,1


In [15]:
from sklearn import set_config

set_config(transform_output="pandas")

In [16]:
# encode the data

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first",sparse_output=False), ["region","day_of_week"])
], remainder="passthrough", n_jobs=-1)

In [17]:
encoder

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``

In [18]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [20]:
import optuna
import tqdm 

In [21]:
# set the experiment

mlflow.set_experiment("Model Selection")

2026/06/13 15:28:33 INFO mlflow.tracking.fluent: Experiment with name 'Model Selection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/38aa807d3665490fbc42be2265bdfad8', creation_time=1781344713767, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781344713767, lifecycle_stage='active', name='Model Selection', tags={}, trace_location=None, workspace='default'>

In [22]:
def objective(trial):
    # start the child run
    with mlflow.start_run(nested=True) as child:
        
        # model name search space
        list_of_models = ["LR", "RF", "GBR", "XGBR"]
        model_name = trial.suggest_categorical("model_name", list_of_models)
    
        if model_name == "LR":
            model = LinearRegression()
    
        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf",10,100,step=10)
            max_depth_rf = trial.suggest_int("max_depth_rf",3,10)
            model = RandomForestRegressor(n_estimators=n_estimators_rf, 
                                          max_depth=max_depth_rf, 
                                          random_state=42, n_jobs=-1)
    
        elif model_name == "GBR":
            n_estimators_gb = trial.suggest_int("n_estimators_gb",10,100,step=10)
            learning_rate_gb = trial.suggest_float("learning_rate_gb",1e-4,1e-1, log=True)
            model = GradientBoostingRegressor(n_estimators=n_estimators_gb, 
                                              learning_rate=learning_rate_gb,
                                             random_state=42)
    
        elif model_name == "XGBR":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,100,step=10)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb",1e-4,1e-1, log=True)
            max_depth_xgb = trial.suggest_int("max_depth_xgb",3,10)
            model = XGBRegressor(n_estimators=n_estimators_xgb,
                                learning_rate=learning_rate_xgb,
                                max_depth=max_depth_xgb)
    
        # log the model name
        mlflow.log_param("model_name",model_name)
        
        # log the model parameters
        mlflow.log_params(model.get_params())
        
        # fit on the data
        model.fit(X_train_encoded,y_train)
    
        # get the predictions
        y_pred = model.predict(X_test_encoded)
    
        # calculate the loss
        loss = mean_absolute_percentage_error(y_test, y_pred)
    
        # log the metric
        mlflow.log_metric("MAPE",loss)
        return loss

In [23]:
# optimize the objective function

with mlflow.start_run(run_name="best_model", nested=True) as parent:

    # create a study object
    study = optuna.create_study(study_name="model_selection", direction="minimize")
    # optimize the objective function
    study.optimize(func=objective, n_trials=50, n_jobs=-1)
    
    # log the best parameters
    mlflow.log_params(study.best_params)
    # log the best error value
    mlflow.log_metric("Best_MAPE", study.best_value)

[I 2026-06-13 15:28:45,027] A new study created in memory with name: model_selection


🏃 View run polite-bird-627 at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/b3bd3fa608a140919cb793a34cca2469
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0
🏃 View run melodic-turtle-812 at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/415a633a5b134f00917846876073c5d7
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0
🏃 View run fun-chimp-46 at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/6aec7e54e1814241884e8ffe2957c3b8
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0


[I 2026-06-13 15:29:55,353] Trial 6 finished with value: 0.30698710068878476 and parameters: {'model_name': 'LR'}. Best is trial 6 with value: 0.30698710068878476.


🏃 View run mercurial-shad-492 at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/de7684a6fc9c42bc9a3cdf47b11e4b47
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0
🏃 View run legendary-loon-46 at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/04a04e5132b14237b5aafd64a2955b98
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0
🏃 View run debonair-jay-714 at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/0909e096a5a244eaa272dd4d8ee2120b
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0
🏃 View run nervous-mouse-178 at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/d985399b23e845478183a31227c5cb63
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/ex

[I 2026-06-13 15:30:15,351] Trial 10 finished with value: 0.30698710068878476 and parameters: {'model_name': 'LR'}. Best is trial 6 with value: 0.30698710068878476.
[I 2026-06-13 15:30:19,423] Trial 3 finished with value: 0.30698710068878476 and parameters: {'model_name': 'LR'}. Best is trial 6 with value: 0.30698710068878476.


...

🏃 View run beautiful-fawn-199 at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/186012009f7541d59a463ae507b7b31e
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0


[I 2026-06-13 15:36:34,306] Trial 49 finished with value: 6.513005311278629 and parameters: {'model_name': 'GBR', 'n_estimators_gb': 100, 'learning_rate_gb': 0.00010490219173724082}. Best is trial 6 with value: 0.30698710068878476.


🏃 View run clean-croc-380 at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/395eaf56605041d889761c37886ef5bd
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0


[I 2026-06-13 15:36:44,321] Trial 45 finished with value: 6.515633557590679 and parameters: {'model_name': 'GBR', 'n_estimators_gb': 100, 'learning_rate_gb': 0.00010060549322302224}. Best is trial 6 with value: 0.30698710068878476.


🏃 View run best_model at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0/runs/eb8fc170a2644c0f8e68ad3092013796
🧪 View experiment at: https://dagshub.com/rajputriya887/uber-demand-prediction.mlflow/#/experiments/0


In [27]:
# best value

study.best_value

0.30698710068878476

In [26]:
# best parameters

study.best_params

{'model_name': 'LR'}

In [28]:
# model value counts

study.trials_dataframe()['params_model_name'].value_counts()

params_model_name
LR      32
GBR      8
XGBR     6
RF       4
Name: count, dtype: int64

In [29]:
from optuna.visualization import (
    plot_optimization_history, 
    plot_parallel_coordinate, 
    plot_param_importances
)

In [30]:
import plotly
plot_optimization_history(study)

In [31]:
plot_parallel_coordinate(study, params=["model_name"])

In [32]:
# train the linear regression model

lr = LinearRegression()

lr.fit(X_train_encoded, y_train)

# get predictions
y_pred_train = lr.predict(X_train_encoded) 
y_pred_test = lr.predict(X_test_encoded)

# loss

mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
mape_test = mean_absolute_percentage_error(y_test, y_pred_test)

print("The training error is ", mape_train)
print("The test error is ", mape_test)

The training error is  0.3174973173589058
The test error is  0.30698710068878476


In [33]:
lr.coef_

array([ 6.59266116, -2.05638952,  1.61048142,  3.5589809 ,  9.08819126,
        2.48639847,  7.8520678 , 10.1342072 , -1.11046656,  8.27602693,
        5.45456779, 10.55872532, -1.47820493,  7.18907985,  6.87749488,
       -1.37837448, -1.70783813, 13.33847201,  5.6738974 ,  3.55876455,
       11.32924195,  5.90374261,  2.88449163, -2.03676138,  2.83674402,
        2.42578855,  6.85780565, -1.94162399, -1.62963472,  0.64633354,
        0.93102079,  1.43743235,  1.61026612,  0.91215046, -0.468891  ,
        1.40961414,  0.49144781,  0.28244361,  0.18601736, -1.42136691])

In [34]:
def tune_ridge(trial):
    # hyperparameter space
    alpha = trial.suggest_float("alpha",30,100)
    
    # make the model object
    ridge = Ridge(alpha=alpha, random_state=42)
    
    # train the model
    ridge.fit(X_train_encoded, y_train)
    
    # get predictions
    y_pred = ridge.predict(X_test_encoded)
    
    # calculate loss
    loss = mean_absolute_percentage_error(y_test, y_pred)

    return loss
        

In [35]:
# create study

study = optuna.create_study(study_name="tune_model", direction="minimize")

[I 2026-06-13 15:42:34,920] A new study created in memory with name: tune_model


In [36]:
# optimize

study.optimize(func=tune_ridge, n_trials=100, n_jobs=-1, show_progress_bar=True)

  0%|          | 0/100 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.306883:   2%|▏         | 2/100 [00:00<00:21,  4.50it/s]

[I 2026-06-13 15:42:42,074] Trial 0 finished with value: 0.3068834690871679 and parameters: {'alpha': 47.837889099666384}. Best is trial 0 with value: 0.3068834690871679.
[I 2026-06-13 15:42:42,097] Trial 2 finished with value: 0.30693799405668976 and parameters: {'alpha': 72.15170813763665}. Best is trial 0 with value: 0.3068834690871679.
[I 2026-06-13 15:42:42,103] Trial 1 finished with value: 0.30690184045643915 and parameters: {'alpha': 58.90590391480911}. Best is trial 0 with value: 0.3068834690871679.


[I 2026-06-13 15:42:42,381] Trial 7 finished with value: 0.3070429187841596 and parameters: {'alpha': 97.64361910237092}. Best is trial 0 with value: 0.3068834690871679.


[I 2026-06-13 15:42:42,253] Trial 4 finished with value: 0.3069351184392668 and parameters: {'alpha': 71.26311825898583}. Best is trial 0 with value: 0.3068834690871679.
[I 2026-06-13 15:42:42,375] Trial 6 finished with value: 0.30688566726232297 and parameters: {'alpha': 49.72384173433535}. Best is trial 0 with value: 0.3068834690871679.
[I 2026-06-13 15:42:42,174] Trial 3 finished with value: 0.30703013233646514 and parameters: {'alpha': 94.97149581559285}. Best is trial 0 with value: 0.3068834690871679.
[I 2026-06-13 15:42:42,410] Trial 5 finished with value: 0.3070062916465192 and parameters: {'alpha': 89.73439463117343}. Best is trial 0 with value: 0.3068834690871679.
[I 2026-06-13 15:42:42,543] Trial 9 finished with value: 0.30687805008833674 and parameters: {'alpha': 38.0715936349782}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:42,625] Trial 12 finished with value: 0.30687907243321716 and parameters: {'alpha': 42.046147330086974}. Best is trial 9 with v

...

[I 2026-06-13 15:42:45,005] Trial 76 finished with value: 0.30687913654337123 and parameters: {'alpha': 33.958237149864644}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,036] Trial 75 finished with value: 0.3068783071312019 and parameters: {'alpha': 36.03982102534768}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,073] Trial 77 finished with value: 0.3068785931220723 and parameters: {'alpha': 35.152537761113535}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,107] Trial 79 finished with value: 0.3068786060508302 and parameters: {'alpha': 35.118210519556484}. Best is trial 9 with value: 0.30687805008833674.


Best trial: 9. Best value: 0.306878:  93%|█████████▎| 93/100 [00:04<00:00, 39.34it/s]

[I 2026-06-13 15:42:45,203] Trial 78 finished with value: 0.306878819190776 and parameters: {'alpha': 34.59716532696562}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,206] Trial 80 finished with value: 0.30687822869005765 and parameters: {'alpha': 36.365716847856135}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,303] Trial 82 finished with value: 0.3068780622609654 and parameters: {'alpha': 37.57103246464327}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,305] Trial 81 finished with value: 0.30687879499489074 and parameters: {'alpha': 41.39933777134693}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,347] Trial 83 finished with value: 0.3068786772905328 and parameters: {'alpha': 41.090621267768384}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,429] Trial 84 finished with value: 0.30687877708693084 and parameters: {'alpha': 41.353791293165145}. Best is tria

Best trial: 9. Best value: 0.306878: 100%|██████████| 100/100 [00:04<00:00, 24.06it/s]

[I 2026-06-13 15:42:45,776] Trial 95 finished with value: 0.30688007171111137 and parameters: {'alpha': 32.49260556846314}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,816] Trial 96 finished with value: 0.30687806569300013 and parameters: {'alpha': 37.519631629003904}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,817] Trial 94 finished with value: 0.30687805415593183 and parameters: {'alpha': 37.71565077135139}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,886] Trial 97 finished with value: 0.3068780654471596 and parameters: {'alpha': 37.52319073577453}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,898] Trial 98 finished with value: 0.3068780637145212 and parameters: {'alpha': 37.54879509703153}. Best is trial 9 with value: 0.30687805008833674.
[I 2026-06-13 15:42:45,937] Trial 99 finished with value: 0.3068781174942373 and parameters: {'alpha': 37.02500445972848}. Best is trial

In [37]:
# best parameters

study.best_params

{'alpha': 38.0715936349782}

In [38]:
# best value

study.best_value

0.30687805008833674